<a href="https://colab.research.google.com/github/palakolanutejasri/project-internvision/blob/main/movie_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning a Large Language Model (LLM) for Sentiment Analysis

Dataset: IMDb Movie Reviews Dataset

Model: DistilBERT

Objective:
To fine-tune a pre-trained DistilBERT model using Transfer Learning for sentiment classification.

In [ ]:
!pip install transformers datasets evaluate accelerate -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)

# What is Transfer Learning?

Transfer Learning is the process of using a pre-trained model and adapting it to a new task.

In this project DistilBERT is already trained on massive text corpora.

We fine-tune it on IMDb reviews instead of training from scratch.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

In [ ]:
print(dataset["train"][0])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import evaluate

In [ ]:
df = pd.DataFrame(
    dataset["train"][:1000]
)

df.head()

In [ ]:
df["label"].value_counts()

In [ ]:
df["label"].value_counts().plot(kind="bar")

plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")

plt.show()

In [ ]:
checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    checkpoint
)

print("Tokenizer Loaded")

In [ ]:
sample = "This movie was absolutely fantastic"

tokenizer(sample)

In [ ]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
small_train = tokenized_dataset[
    "train"
].shuffle(seed=42).select(range(2000))

small_test = tokenized_dataset[
    "test"
].shuffle(seed=42).select(range(500))

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

print("Model Loaded")

In [ ]:
accuracy_metric = evaluate.load("accuracy")

f1_metric = evaluate.load("f1")

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

In [ ]:
training_args = TrainingArguments(

    output_dir="./results",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    evaluation_strategy="epoch",

    save_strategy="epoch"
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics
)

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

In [ ]:
training_args = TrainingArguments(

    output_dir="./results",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    evaluation_strategy="epoch",

    save_strategy="epoch"
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics
)

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
import transformers
print(transformers.__version__)
print(transformers.__file__)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()

print(results)

In [ ]:
print("Accuracy :", results.get("eval_accuracy"))
print("F1 Score :", results.get("eval_f1"))

In [ ]:
predictions = trainer.predict(small_test)

predictions

In [ ]:
import numpy as np

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_true = predictions.label_ids

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_true,
    y_pred
)

cm

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm
)

disp.plot()

plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred
    )
)

In [ ]:
import torch

def predict_sentiment(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    outputs = model(**inputs)

    prediction = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

    if prediction == 1:
        return "Positive"

    else:
        return "Negative"

In [ ]:
predict_sentiment(
    "I absolutely loved this movie. Amazing acting and story."
)

In [ ]:
predict_sentiment(
    "Worst movie ever. Waste of time."
)

In [ ]:
model.save_pretrained(
    "fine_tuned_distilbert"
)

tokenizer.save_pretrained(
    "fine_tuned_distilbert"
)

print("Model Saved Successfully")

Conclusion

A pre-trained DistilBERT model was fine-tuned on the IMDb Movie Review Dataset for sentiment classification.

Transfer Learning was applied by leveraging the knowledge already learned by DistilBERT from large text corpora.

The fine-tuned model successfully classified movie reviews as Positive or Negative and achieved good performance on unseen test data.

This project demonstrates the practical use of Large Language Models (LLMs) in Natural Language Processing tasks.